In [1]:
import os
import numpy as np
import pandas as pd
import mikeio
from scipy.interpolate import RegularGridInterpolator

# 3D salinity

In [2]:
# =============================================================================
# 1. 全局参数配置 (Global Parameters)
# =============================================================================

# 输入 DFS3 文件路径
DFS3_FILE = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_S_qc\HYCOM_2025_salinity_ALL_small.dfs3"

# 输出文件夹路径 (请确保该文件夹存在)
OUTPUT_DIR = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_S_qc"

# 基础坐标偏移量
LON0, LAT0 = 117.68, 23.88

# 剖面水平间距 (单位：度)
DX = 0.01

# 插值方式：'linear' 或 'nearest'
INTERP_MODE = "linear"


In [3]:
# =============================================================================
# 2. 边界坐标配置 (Boundary Coordinates Configuration)
# =============================================================================

BOUNDARIES = {
    "N": {
        "start_lon": 118.861559618834,  
        "start_lat": 24.440174188642,  
        "end_lon":   118.683639625575,  
        "end_lat":   24.6305220196772   
    },
    "E": {
        "start_lon": 118.24760732877,  
        "start_lat": 23.9406130942768,  
        "end_lon":   118.861559618834,  
        "end_lat":   24.440174188642   
    },
    "S": {
        "start_lon": 118.020647519105,  
        "start_lat": 24.2027468061256,  
        "end_lon":   118.24760732877,  
        "end_lat":   23.9406130942768   
    }
}

In [4]:
def extract_boundary_profile(dfs3_path, out_dfs2_path, bnd_name, bnd_coords, lon0, lat0, dx, interp_mode):
    """
    从 DFS3 文件中提取单个剖面，自动填充水平边缘的空值，并保存为 DFS2 文件。
    同时增加 QC：
    - 若某个时间步中任意位置出现值 = 20，则判定该时间步为错误数据
    - 输出错误时间步索引
    - 用前后时间步做线性插值替换该时间步
    """
    print(f"\n[{bnd_name} 边界] 开始处理...")
    
    if any(v is None for v in bnd_coords.values()):
        print(f"[{bnd_name} 边界] 错误: 坐标未填写完整，已跳过。")
        return

    # 读取 DFS3 网格信息
    dfs = mikeio.open(dfs3_path)
    ds = dfs.read()
    geom = dfs.geometry
    z = np.asarray(geom.z, dtype=float)
    x = np.round((lon0 + np.asarray(geom.x, dtype=float)), 2)
    y = np.round((lat0 + np.asarray(geom.y, dtype=float)), 2)

    # 计算采样剖面点
    start_lon, start_lat = bnd_coords["start_lon"], bnd_coords["start_lat"]
    end_lon, end_lat = bnd_coords["end_lon"], bnd_coords["end_lat"]
    
    dx_total = end_lon - start_lon
    dy_total = end_lat - start_lat
    distance_total = np.sqrt(dx_total**2 + dy_total**2)
    
    npt = int(np.floor(distance_total / dx)) + 1
    npt = max(npt, 2)
    
    section_x = np.linspace(start_lon, end_lon, npt)
    section_y = np.linspace(start_lat, end_lat, npt)
    section_dist = np.linspace(0.0, distance_total, npt)
    
    print(f"[{bnd_name} 边界] 剖面长度: {distance_total:.4f} 度, 采样点数: {npt}")
    
    sample_points = np.column_stack([section_y, section_x]) 
    
    n_time = ds.n_timesteps
    n_item = ds.n_items
    nz = geom.nz
    section_arrays = []

    # 执行空间插值提取
    for item_idx in range(n_item):
        item_name = ds.items[item_idx].name
        print(f"[{bnd_name} 边界] 正在提取变量 {item_name} ({item_idx+1}/{n_item})")
        
        data3d = ds[item_idx].to_numpy()
        out = np.full((n_time, nz, npt), np.nan, dtype=float)

        for t in range(n_time):
            layer3d = data3d[t]
            for k in range(nz):
                layer2d = layer3d[k, :, :]
                
                # 构建插值器
                interp_func = RegularGridInterpolator(
                    (y, x), layer2d, method=interp_mode, bounds_error=False, fill_value=np.nan
                )
                
                # 提取当前水平剖面数据
                profile_1d = interp_func(sample_points)
                
                # 水平方向空值填充
                if np.isnan(profile_1d).any() and not np.isnan(profile_1d).all():
                    profile_series = pd.Series(profile_1d)
                    profile_1d = profile_series.ffill().bfill().to_numpy()
                
                out[t, k, :] = profile_1d
                
            if (t + 1) % 500 == 0 or (t + 1) == n_time:
                print(f"  --> 时间步进度: {t + 1}/{n_time}")

        # ==========================================================
        # QC: 检查时间步中是否存在值 = 20 的错误数据
        # 规则：某个时间步任意位置出现 20，即认为该时间步有问题
        # 处理：整步置为 NaN，然后沿时间轴线性插值
        # ==========================================================
        bad_time_mask = np.any(np.isclose(out, 20.0, rtol=0.0, atol=0.0), axis=(1, 2))
        bad_times = np.where(bad_time_mask)[0]

        if bad_times.size > 0:
            print(f"[{bnd_name} 边界][QC] 检测到错误时间步（含值=20）：{bad_times.tolist()}")

            # 将这些时间步整体置为 NaN，准备按时间轴插值
            out[bad_time_mask, :, :] = np.nan

            # 将 3D 数组展平成二维：time x space
            flat = out.reshape(n_time, -1)

            # 沿时间轴做线性插值
            flat_df = pd.DataFrame(flat)
            flat_df = flat_df.interpolate(axis=0, method="linear", limit_direction="both")

            # 若首尾仍有 NaN（例如前几步或后几步连续坏值），用最近有效值补齐
            flat_df = flat_df.ffill().bfill()

            out = flat_df.to_numpy().reshape(n_time, nz, npt)

        section_arrays.append(out)

    # 构建导出 DFS2 网格
    grid2 = mikeio.Grid2D(x=section_dist, y=z, projection=geom.projection)
    
    data_arrays = []
    for item_idx in range(n_item):
        da = mikeio.DataArray(
            data=section_arrays[item_idx],
            geometry=grid2,
            time=ds.time,
            item=ds.items[item_idx]
        )
        data_arrays.append(da)

    ds_out = mikeio.Dataset(data_arrays)
    ds_out.to_dfs(out_dfs2_path)
    print(f"[{bnd_name} 边界] 提取完成，文件已保存至: {out_dfs2_path}")

In [5]:
# =============================================================================
# 4. 批量执行逻辑 (Batch Execution)
# =============================================================================

if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    for bnd_name, bnd_coords in BOUNDARIES.items():
        out_filename = f"HYCOM_2025_salinity_{bnd_name}.dfs2"
        out_path = os.path.join(OUTPUT_DIR, out_filename)
        
        extract_boundary_profile(
            dfs3_path=DFS3_FILE,
            out_dfs2_path=out_path,
            bnd_name=bnd_name,
            bnd_coords=bnd_coords,
            lon0=LON0,
            lat0=LAT0,
            dx=DX,
            interp_mode=INTERP_MODE
        )
    
    print("\n所有配置的边界提取与边缘值填充流程已结束。")


[N 边界] 开始处理...
[N 边界] 剖面长度: 0.2606 度, 采样点数: 27
[N 边界] 正在提取变量 Salinity (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[N 边界][QC] 检测到错误时间步（含值=20）：[1656]
[N 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_S_qc\HYCOM_2025_salinity_N.dfs2

[E 边界] 开始处理...
[E 边界] 剖面长度: 0.7915 度, 采样点数: 80
[E 边界] 正在提取变量 Salinity (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[E 边界][QC] 检测到错误时间步（含值=20）：[1656]
[E 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_S_qc\HYCOM_2025_salinity_E.dfs2

[S 边界] 开始处理...
[S 边界] 剖面长度: 0.3467 度, 采样点数: 35
[S 边界] 正在提取变量 Salinity (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[S 边界][QC] 检测到错误时间步（含值=20）：[1656]
[S 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_

# 3D temperature

In [6]:
# =============================================================================
# 1. 全局参数配置 (Global Parameters)
# =============================================================================

# 输入 DFS3 文件路径
DFS3_FILE = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_T_qc\HYCOM_2025_water_temp_ALL_small.dfs3"

# 输出文件夹路径 (请确保该文件夹存在)
OUTPUT_DIR = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_T_qc"

# 基础坐标偏移量
LON0, LAT0 = 117.68, 23.88

# 剖面水平间距 (单位：度)
DX = 0.01

# 插值方式：'linear' 或 'nearest'
INTERP_MODE = "linear"


In [7]:
# =============================================================================
# 4. 批量执行逻辑 (Batch Execution)
# =============================================================================

if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    for bnd_name, bnd_coords in BOUNDARIES.items():
        out_filename = f"HYCOM_2025_water_temp_{bnd_name}.dfs2"
        out_path = os.path.join(OUTPUT_DIR, out_filename)
        
        extract_boundary_profile(
            dfs3_path=DFS3_FILE,
            out_dfs2_path=out_path,
            bnd_name=bnd_name,
            bnd_coords=bnd_coords,
            lon0=LON0,
            lat0=LAT0,
            dx=DX,
            interp_mode=INTERP_MODE
        )
    
    print("\n所有配置的边界提取与边缘值填充流程已结束。")


[N 边界] 开始处理...
[N 边界] 剖面长度: 0.2606 度, 采样点数: 27
[N 边界] 正在提取变量 Temperature (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[N 边界][QC] 检测到错误时间步（含值=20）：[135, 139, 167, 1071, 1247, 1311, 1343, 1351, 1363, 1379, 1399, 1656]
[N 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_T_qc\HYCOM_2025_water_temp_N.dfs2

[E 边界] 开始处理...
[E 边界] 剖面长度: 0.7915 度, 采样点数: 80
[E 边界] 正在提取变量 Temperature (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[E 边界][QC] 检测到错误时间步（含值=20）：[135, 139, 167, 1071, 1247, 1311, 1343, 1351, 1363, 1379, 1399]
[E 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_T_qc\HYCOM_2025_water_temp_E.dfs2

[S 边界] 开始处理...
[S 边界] 剖面长度: 0.3467 度, 采样点数: 35
[S 边界] 正在提取变量 Temperature (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度

# 3D U

In [8]:
# =============================================================================
# 1. 全局参数配置 (Global Parameters)
# =============================================================================

# 输入 DFS3 文件路径
DFS3_FILE = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_ALL_small.dfs3"

# 输出文件夹路径 (请确保该文件夹存在)
OUTPUT_DIR = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc"

# 基础坐标偏移量
LON0, LAT0 = 117.68, 23.88

# 剖面水平间距 (单位：度)
DX = 0.01

# 插值方式：'linear' 或 'nearest'
INTERP_MODE = "linear"


In [9]:
# =============================================================================
# 4. 批量执行逻辑 (Batch Execution)
# =============================================================================

if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    for bnd_name, bnd_coords in BOUNDARIES.items():
        out_filename = f"HYCOM_2025_U_{bnd_name}.dfs2"
        out_path = os.path.join(OUTPUT_DIR, out_filename)
        
        extract_boundary_profile(
            dfs3_path=DFS3_FILE,
            out_dfs2_path=out_path,
            bnd_name=bnd_name,
            bnd_coords=bnd_coords,
            lon0=LON0,
            lat0=LAT0,
            dx=DX,
            interp_mode=INTERP_MODE
        )
    
    print("\n所有配置的边界提取与边缘值填充流程已结束。")


[N 边界] 开始处理...
[N 边界] 剖面长度: 0.2606 度, 采样点数: 27
[N 边界] 正在提取变量 U (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[N 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_N.dfs2

[E 边界] 开始处理...
[E 边界] 剖面长度: 0.7915 度, 采样点数: 80
[E 边界] 正在提取变量 U (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[E 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_E.dfs2

[S 边界] 开始处理...
[S 边界] 剖面长度: 0.3467 度, 采样点数: 35
[S 边界] 正在提取变量 U (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[S 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_U_qc\HYCOM_2025_U_S.dfs2

所有配置的边界提取与边缘值填充流程已结束。


In [10]:
# =============================================================================
# 1. 全局参数配置 (Global Parameters)
# =============================================================================

# 输入 DFS3 文件路径
DFS3_FILE = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_ALL_small.dfs3"

# 输出文件夹路径 (请确保该文件夹存在)
OUTPUT_DIR = r"C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc"

# 基础坐标偏移量
LON0, LAT0 = 117.68, 23.88

# 剖面水平间距 (单位：度)
DX = 0.01

# 插值方式：'linear' 或 'nearest'
INTERP_MODE = "linear"


In [11]:
# =============================================================================
# 4. 批量执行逻辑 (Batch Execution)
# =============================================================================

if __name__ == "__main__":
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    for bnd_name, bnd_coords in BOUNDARIES.items():
        out_filename = f"HYCOM_2025_V_{bnd_name}.dfs2"
        out_path = os.path.join(OUTPUT_DIR, out_filename)
        
        extract_boundary_profile(
            dfs3_path=DFS3_FILE,
            out_dfs2_path=out_path,
            bnd_name=bnd_name,
            bnd_coords=bnd_coords,
            lon0=LON0,
            lat0=LAT0,
            dx=DX,
            interp_mode=INTERP_MODE
        )
    
    print("\n所有配置的边界提取与边缘值填充流程已结束。")


[N 边界] 开始处理...
[N 边界] 剖面长度: 0.2606 度, 采样点数: 27
[N 边界] 正在提取变量 V (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[N 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_N.dfs2

[E 边界] 开始处理...
[E 边界] 剖面长度: 0.7915 度, 采样点数: 80
[E 边界] 正在提取变量 V (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[E 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_E.dfs2

[S 边界] 开始处理...
[S 边界] 剖面长度: 0.3467 度, 采样点数: 35
[S 边界] 正在提取变量 V (1/1)
  --> 时间步进度: 500/2920
  --> 时间步进度: 1000/2920
  --> 时间步进度: 1500/2920
  --> 时间步进度: 2000/2920
  --> 时间步进度: 2500/2920
  --> 时间步进度: 2920/2920
[S 边界] 提取完成，文件已保存至: C:\Project\2025_HT\Input\HYCOM_2025_monthly_V_qc\HYCOM_2025_V_S.dfs2

所有配置的边界提取与边缘值填充流程已结束。
